## Running RAG Pipeline

In [ ]:
# NOTE: To run the RAG pipeline, we'd use the following.
# 1. QDrant vector database
# 2. Docker container to run the QDrant database image locally on the machine.
# 3. OpenAI embeddings API to generate vector embeddings for the user query.
# 4. Claude API to generate the final response based on the retrieved content
#    from QDrant database.
#
# Before running the RAG pipeline, run "loading_vectors.ipynb" to load the vector
# embeddings into the QDrant database.

### Installing Libraries

In [ ]:
%pip install anthropic python-dotenv openai qdrant-client

### Setting up the Environment

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")
claude_model = os.getenv("CLAUDE_MODEL")
openai_api_key = os.getenv("OPENAI_API_KEY")

QDRANT_URL = "http://localhost:6333"
COLLECTION_NAME = "Travel_guide_for_India"

### Creating the Anthropic Client

In [ ]:
import anthropic

anthropic_client = anthropic.Anthropic(api_key=anthropic_api_key)

### Creating the OpenAI Client

In [ ]:
from openai import OpenAI

openai_client = OpenAI(api_key=openai_api_key)

### Creating Vector Embedding Generator Function

In [ ]:
def generate_vector_embeddings(user_query):

    response = openai_client.embeddings.create(
        model="text-embedding-3-small",
        input=user_query
    )

    return response.data[0].embedding

### Creating Context Retrieval Function

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct


def context_retrieval(user_query):

    # Generating Vector Embeddings for the User Query
    user_embeddings = generate_vector_embeddings(user_query)

    # Initializing Qdrant client
    qdrant_client = QdrantClient(url=QDRANT_URL)

    # Throwing a request to QDrantDB to retrieve the top-matched content along with the vector embeddings
    search_result = qdrant_client.query_points(
        collection_name="Travel_guide_for_India",
        query=user_embeddings,
        with_vectors=True,
        with_payload=True,
        limit=2
    ).points

    # Retrieving the top text content that can be used to answer the user query
    supporting_text_content = search_result[0].payload.get("content")
    return supporting_text_content

### Querying with Retrieved Context

In [ ]:
user_query = """Which months are the best to visit India?"""

context = context_retrieval(user_query)

with anthropic_client.messages.stream(
    model=claude_model,
    system="You are a helpful AI Chatbot",
    messages=[
        {
            "role": "user",
            "content":  f"""answer the user query using the provided supporting knowledge \n 
                                            user query: {user_query} \n
                                            supporting_text: {context}""",
        }
    ],
    max_tokens=10000
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)